In [ ]:
"""
UNSW-NB15 Cross-Dataset Validation Experiment
Otegen Danial | Astana IT University | 2026

Run this in Google Colab or Jupyter Notebook.

Step 1: Download dataset
    from google.colab import files  # if Colab
    # OR manually download from:
    # https://www.kaggle.com/datasets/mrwellsdavid/unsw-nb15
    # Files needed: UNSW_NB15_training-set.csv, UNSW_NB15_testing-set.csv

Step 2: Run this script
"""

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, log_loss, cohen_kappa_score,
    matthews_corrcoef, balanced_accuracy_score, hamming_loss
)
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────────────────────

# Option A: Kaggle (recommended)
# !pip install kaggle
# !kaggle datasets download -d mrwellsdavid/unsw-nb15
# !unzip unsw-nb15.zip

# Option B: Direct from UNSW (requires registration)
# https://research.unsw.edu.au/projects/unsw-nb15-dataset

# Load training and test sets
TRAIN_PATH = '/content/UNSW_NB15_training-set.csv'
TEST_PATH  = '/content/UNSW_NB15_testing-set.csv'

print("Loading UNSW-NB15...")
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")
print(f"\nTrain label distribution:\n{train_df['label'].value_counts()}")
print(f"\nAttack categories (train):\n{train_df['attack_cat'].value_counts()}")

# ─────────────────────────────────────────────────────────────
# 2. PREPROCESSING
# ─────────────────────────────────────────────────────────────

def preprocess(train, test):
    # Drop ID columns if present
    drop_cols = ['id', 'attack_cat']
    train = train.drop(columns=[c for c in drop_cols if c in train.columns])
    test  = test.drop(columns=[c for c in drop_cols if c in test.columns])

    # Target: binary (0=normal, 1=attack) — same as NSL-KDD binary setup
    X_train = train.drop('label', axis=1)
    y_train = train['label']
    X_test  = test.drop('label', axis=1)
    y_test  = test['label']

    # Encode categorical columns
    cat_cols = X_train.select_dtypes(include=['object']).columns
    le = LabelEncoder()
    for col in cat_cols:
        combined = pd.concat([X_train[col], X_test[col]], axis=0)
        le.fit(combined)
        X_train[col] = le.transform(X_train[col])
        X_test[col]  = le.transform(X_test[col])

    # Fill any NaN
    X_train = X_train.fillna(0)
    X_test  = X_test.fillna(0)

    # Normalize (fit on train only)
    scaler = MinMaxScaler()
    X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
    X_test  = pd.DataFrame(scaler.transform(X_test),      columns=X_test.columns)

    return X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = preprocess(train_df.copy(), test_df.copy())
print(f"\nAfter preprocessing:")
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

# ─────────────────────────────────────────────────────────────
# 3. MODELS
# ─────────────────────────────────────────────────────────────

models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'KNN':           KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost':       XGBClassifier(
                         n_estimators=100, random_state=42,
                         use_label_encoder=False,
                         eval_metric='logloss',
                         n_jobs=-1, verbosity=0
                     ),
}

# ─────────────────────────────────────────────────────────────
# 4. EVALUATE — same 10 metrics as NSL-KDD paper
# ─────────────────────────────────────────────────────────────

def evaluate(model, X_tr, X_te, y_tr, y_te, name):
    print(f"\nTraining {name}...")
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)

    # Probabilities for Log Loss and ROC AUC
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_te)[:, 1]
    else:
        y_prob = y_pred

    results = {
        'Model':             name,
        'Accuracy':          round(accuracy_score(y_te, y_pred), 4),
        'Precision':         round(precision_score(y_te, y_pred, average='weighted', zero_division=0), 4),
        'Recall':            round(recall_score(y_te, y_pred, average='weighted', zero_division=0), 4),
        'F1-Score':          round(f1_score(y_te, y_pred, average='weighted', zero_division=0), 4),
        'ROC AUC':           round(roc_auc_score(y_te, y_prob), 4),
        'Log Loss':          round(log_loss(y_te, model.predict_proba(X_te) if hasattr(model,'predict_proba') else y_pred), 4),
        "Cohen's Kappa":     round(cohen_kappa_score(y_te, y_pred), 4),
        'MCC':               round(matthews_corrcoef(y_te, y_pred), 4),
        'Balanced Accuracy': round(balanced_accuracy_score(y_te, y_pred), 4),
        'Hamming Loss':      round(hamming_loss(y_te, y_pred), 4),
    }

    for k, v in results.items():
        if k != 'Model':
            print(f"  {k}: {v}")

    return results

all_results = []
for name, model in models.items():
    r = evaluate(model, X_train, X_test, y_train, y_test, name)
    all_results.append(r)

# ─────────────────────────────────────────────────────────────
# 5. RESULTS TABLES
# ─────────────────────────────────────────────────────────────

results_df = pd.DataFrame(all_results)
results_df = results_df.set_index('Model')

print("\n" + "="*80)
print("TABLE: Standard Classification Metrics (UNSW-NB15)")
print("="*80)
std_cols = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
print(results_df[std_cols].to_string())

print("\n" + "="*80)
print("TABLE: Advanced Evaluation Metrics (UNSW-NB15)")
print("="*80)
adv_cols = ['ROC AUC', 'Log Loss', "Cohen's Kappa", 'MCC', 'Balanced Accuracy', 'Hamming Loss']
print(results_df[adv_cols].to_string())

# Save to CSV for paper
results_df.to_csv('unsw_nb15_results.csv')
print("\nResults saved to unsw_nb15_results.csv")

# ─────────────────────────────────────────────────────────────
# 6. CROSS-DATASET COMPARISON TABLE
# ─────────────────────────────────────────────────────────────

# Paste your NSL-KDD results here (from your existing paper)
nsl_kdd_results = {
    'Decision Tree': {'Accuracy': 0.7615, 'Precision': 0.7956, 'ROC AUC': 0.8027, 'Log Loss': 8.597},
    'KNN':           {'Accuracy': 0.7568, 'Precision': 0.8087, 'ROC AUC': 0.8200, 'Log Loss': 7.883},
    'Random Forest': {'Accuracy': 0.7517, 'Precision': 0.8156, 'ROC AUC': 0.9326, 'Log Loss': 3.145},
    'XGBoost':       {'Accuracy': 0.7574, 'Precision': 0.7975, 'ROC AUC': 0.9596, 'Log Loss': 2.022},
}

print("\n" + "="*80)
print("CROSS-DATASET COMPARISON: NSL-KDD vs UNSW-NB15 (ROC AUC)")
print("="*80)
print(f"{'Model':<20} {'NSL-KDD ROC AUC':>18} {'UNSW-NB15 ROC AUC':>20}")
print("-"*60)
for model_name in ['Decision Tree', 'KNN', 'Random Forest', 'XGBoost']:
    nsl = nsl_kdd_results[model_name]['ROC AUC']
    unsw = results_df.loc[model_name, 'ROC AUC'] if model_name in results_df.index else 'N/A'
    print(f"{model_name:<20} {nsl:>18.4f} {unsw:>20.4f}")

# ─────────────────────────────────────────────────────────────
# 7. PLOTS
# ─────────────────────────────────────────────────────────────

import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # non-interactive backend

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

model_names = results_df.index.tolist()

# Plot 1: Accuracy, Precision, Recall, F1
x = np.arange(len(model_names))
width = 0.2
ax = axes[0]
for i, metric in enumerate(['Accuracy', 'Precision', 'Recall', 'F1-Score']):
    vals = [results_df.loc[m, metric] for m in model_names]
    ax.bar(x + i*width, vals, width, label=metric)
ax.set_xticks(x + width*1.5)
ax.set_xticklabels(model_names, rotation=15, ha='right')
ax.set_ylim(0.7, 1.0)
ax.set_title('Classification Metrics (UNSW-NB15)')
ax.legend(fontsize=8)
ax.set_ylabel('Score')

# Plot 2: ROC AUC + Log Loss comparison NSL-KDD vs UNSW-NB15
ax = axes[1]
nsl_auc  = [nsl_kdd_results[m]['ROC AUC'] for m in model_names]
unsw_auc = [results_df.loc[m, 'ROC AUC'] for m in model_names]
x = np.arange(len(model_names))
ax.bar(x - 0.2, nsl_auc,  0.35, label='NSL-KDD',    color='steelblue')
ax.bar(x + 0.2, unsw_auc, 0.35, label='UNSW-NB15',  color='darkorange')
ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=15, ha='right')
ax.set_ylim(0.7, 1.0)
ax.set_title('ROC AUC: Cross-Dataset Comparison')
ax.legend()
ax.set_ylabel('ROC AUC')

# Plot 3: MCC, Balanced Accuracy, Cohen's Kappa
ax = axes[2]
for i, metric in enumerate(['MCC', 'Balanced Accuracy', "Cohen's Kappa"]):
    vals = [results_df.loc[m, metric] for m in model_names]
    ax.bar(x + i*width, vals, width, label=metric)
ax.set_xticks(x + width)
ax.set_xticklabels(model_names, rotation=15, ha='right')
ax.set_title('Advanced Metrics (UNSW-NB15)')
ax.legend(fontsize=8)
ax.set_ylabel('Score')

plt.tight_layout()
plt.savefig('unsw_nb15_results.png', dpi=150, bbox_inches='tight')
print("Plot saved to unsw_nb15_results.png")

print("\nDONE. Upload unsw_nb15_results.csv and unsw_nb15_results.png to your paper.")

Loading UNSW-NB15...
Train shape: (82332, 45)
Test shape:  (175341, 45)

Train label distribution:
label
1    45332
0    37000
Name: count, dtype: int64

Attack categories (train):
attack_cat
Normal            37000
Generic           18871
Exploits          11132
Fuzzers            6062
DoS                4089
Reconnaissance     3496
Analysis            677
Backdoor            583
Shellcode           378
Worms                44
Name: count, dtype: int64

After preprocessing:
X_train: (82332, 42), X_test: (175341, 42)

Training Decision Tree...
  Accuracy: 0.896
  Precision: 0.9135
  Recall: 0.896
  F1-Score: 0.8986
  ROC AUC: 0.9145
  Log Loss: 3.7458
  Cohen's Kappa: 0.7761
  MCC: 0.7885
  Balanced Accuracy: 0.9145
  Hamming Loss: 0.104

Training KNN...
  Accuracy: 0.8594
  Precision: 0.8926
  Recall: 0.8594
  F1-Score: 0.8637
  ROC AUC: 0.9401
  Log Loss: 2.3321
  Cohen's Kappa: 0.7059
  MCC: 0.7291
  Balanced Accuracy: 0.888
  Hamming Loss: 0.1406

Training Random Forest...
  Accura

In [ ]:
# ─────────────────────────────────────────────────────────────
# 8. ROC CURVES — Cross-Dataset Comparison
# ─────────────────────────────────────────────────────────────

from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colors = {
    'Decision Tree': 'blue',
    'KNN':           'orange',
    'Random Forest': 'green',
    'XGBoost':       'red',
}

# ── Левая панель: UNSW-NB15 (у нас есть y_test и модели) ──
ax = axes[0]
for name, model in models.items():
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_val = results_df.loc[name, 'ROC AUC']
    ax.plot(fpr, tpr, color=colors[name], linewidth=2,
            label=f'{name} (AUC = {auc_val:.4f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random classifier')
ax.set_xlabel('False Positive Rate (FPR)', fontsize=11)
ax.set_ylabel('True Positive Rate (TPR)', fontsize=11)
ax.set_title('ROC Curves — UNSW-NB15', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.02])
ax.grid(True, alpha=0.3)

# ── Правая панель: NSL-KDD (используем только AUC значения — approx кривые) ──
# Если у тебя нет сохранённых NSL-KDD моделей, рисуем только точки AUC
# Если есть — загрузи модели и добавь roc_curve аналогично

ax = axes[1]

# Приближённые кривые на основе известных AUC значений
# (реальные кривые нужны только если у тебя есть NSL-KDD y_test + predict_proba)
nsl_auc_vals = {
    'Decision Tree': 0.8027,
    'KNN':           0.8200,
    'Random Forest': 0.9326,
    'XGBoost':       0.9596,
}

# Если у тебя есть NSL-KDD модели и y_test_nsl — замени этот блок на реальные кривые
# Иначе — используем Bar chart AUC вместо кривых на этой панели
model_names_list = list(nsl_auc_vals.keys())
auc_values = list(nsl_auc_vals.values())
bar_colors = [colors[m] for m in model_names_list]

bars = ax.bar(model_names_list, auc_values, color=bar_colors, alpha=0.8, edgecolor='black')
for bar, val in zip(bars, auc_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylim(0.7, 1.0)
ax.set_ylabel('ROC AUC', fontsize=11)
ax.set_title('ROC AUC — NSL-KDD KDDTest+', fontsize=13, fontweight='bold')
ax.set_xticklabels(model_names_list, rotation=15, ha='right')
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=0.9, color='gray', linestyle='--', linewidth=1, alpha=0.7)

plt.suptitle('Cross-Dataset ROC Comparison: UNSW-NB15 vs NSL-KDD KDDTest+',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('roc_curves_cross_dataset.png', dpi=150, bbox_inches='tight')
print("ROC curves saved to roc_curves_cross_dataset.png")

ROC curves saved to roc_curves_cross_dataset.png
